In [ ]:
# ============================================================
# SEQUENCE-TO-SEQUENCE (Seq2Seq) MACHINE TRANSLATION
# English to French using TensorFlow/Keras and LSTM
# ============================================================

import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------------------------
# 1. PREPARE THE DATASET
# ------------------------------------------------------------

english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "good morning",
    "good night",
    "i love you",
    "what is your name",
    "where are you",
    "i am happy",
    "i am learning",
    "see you later",
    "goodbye",
    "how are you doing",
    "i am hungry",
    "i am tired",
    "please help me",
    "come here",
    "open the door",
    "close the door"
]

french_sentences = [
    "bonjour",
    "comment allez vous",
    "je vais bien",
    "merci",
    "bonjour",
    "bonne nuit",
    "je t aime",
    "comment vous appelez vous",
    "ou etes vous",
    "je suis heureux",
    "j apprends",
    "a plus tard",
    "au revoir",
    "comment allez vous",
    "j ai faim",
    "je suis fatigue",
    "aidez moi",
    "viens ici",
    "ouvre la porte",
    "ferme la porte"
]

# Add start and end tokens to French sentences
french_sentences = [
    "<start> " + sentence + " <end>"
    for sentence in french_sentences
]

print("Number of training examples:", len(english_sentences))


# ------------------------------------------------------------
# 2. TOKENIZATION
# Convert words into numerical IDs
# ------------------------------------------------------------

english_tokenizer = Tokenizer(
    lower=True,
    oov_token="<unk>"
)

french_tokenizer = Tokenizer(
    lower=True,
    oov_token="<unk>"
)

english_tokenizer.fit_on_texts(english_sentences)
french_tokenizer.fit_on_texts(french_sentences)

encoder_sequences = english_tokenizer.texts_to_sequences(
    english_sentences
)

decoder_sequences = french_tokenizer.texts_to_sequences(
    french_sentences
)

english_vocab_size = len(english_tokenizer.word_index) + 1
french_vocab_size = len(french_tokenizer.word_index) + 1

print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)


# ------------------------------------------------------------
# 3. PADDING
# Make all sequences the same length
# ------------------------------------------------------------

max_encoder_length = max(
    len(sequence) for sequence in encoder_sequences
)

max_decoder_length = max(
    len(sequence) for sequence in decoder_sequences
)

encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_length,
    padding="post"
)

decoder_full_data = pad_sequences(
    decoder_sequences,
    maxlen=max_decoder_length,
    padding="post"
)

# Teacher forcing:
# Decoder input:  <start> bonjour
# Decoder target: bonjour <end>

decoder_input_data = decoder_full_data[:, :-1]
decoder_target_data = decoder_full_data[:, 1:]

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


# ------------------------------------------------------------
# 4. BUILD THE ENCODER
# ------------------------------------------------------------

embedding_dim = 64
latent_dim = 128

encoder_inputs = Input(
    shape=(None,),
    name="encoder_inputs"
)

encoder_embedding_layer = Embedding(
    input_dim=english_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)

encoder_embedding = encoder_embedding_layer(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


# ------------------------------------------------------------
# 5. BUILD THE DECODER
# ------------------------------------------------------------

decoder_inputs = Input(
    shape=(None,),
    name="decoder_inputs"
)

decoder_embedding_layer = Embedding(
    input_dim=french_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    french_vocab_size,
    activation="softmax",
    name="word_prediction"
)

decoder_outputs = decoder_dense(decoder_outputs)


# ------------------------------------------------------------
# 6. CREATE THE COMPLETE SEQ2SEQ MODEL
# ------------------------------------------------------------

model = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs
)

model.summary()


# ------------------------------------------------------------
# 7. COMPILE THE MODEL
# ------------------------------------------------------------

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["sparse_categorical_accuracy"]
)


# ------------------------------------------------------------
# 8. TRAIN THE MODEL
# ------------------------------------------------------------

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=4,
    epochs=500,
    verbose=1
)


# ------------------------------------------------------------
# 9. CREATE THE INFERENCE ENCODER
# ------------------------------------------------------------

encoder_model = Model(
    inputs=encoder_inputs,
    outputs=encoder_states
)


# ------------------------------------------------------------
# 10. CREATE THE INFERENCE DECODER
# ------------------------------------------------------------

decoder_state_input_h = Input(
    shape=(latent_dim,),
    name="decoder_state_h"
)

decoder_state_input_c = Input(
    shape=(latent_dim,),
    name="decoder_state_c"
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_single_input = Input(
    shape=(1,),
    name="decoder_token_input"
)

decoder_single_embedding = decoder_embedding_layer(
    decoder_single_input
)

decoder_single_outputs, state_h, state_c = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs
)

decoder_single_outputs = decoder_dense(decoder_single_outputs)

decoder_model = Model(
    inputs=[
        decoder_single_input,
        decoder_state_input_h,
        decoder_state_input_c
    ],
    outputs=[
        decoder_single_outputs,
        state_h,
        state_c
    ]
)


# ------------------------------------------------------------
# 11. TRANSLATION FUNCTION
# ------------------------------------------------------------

start_token_id = french_tokenizer.word_index["<start>"]
end_token_id = french_tokenizer.word_index["<end>"]


def translate_sentence(sentence):

    # Convert English text to token IDs
    input_sequence = english_tokenizer.texts_to_sequences(
        [sentence.lower()]
    )

    # Pad the input
    input_sequence = pad_sequences(
        input_sequence,
        maxlen=max_encoder_length,
        padding="post"
    )

    # Encode the English sentence
    states_value = encoder_model.predict(
        input_sequence,
        verbose=0
    )

    # Start the decoder
    target_seq = np.array([[start_token_id]])

    translated_words = []

    # Generate tokens one by one
    for _ in range(max_decoder_length):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Select the most probable token
        sampled_token_id = int(
            np.argmax(output_tokens[0, -1, :])
        )

        # Stop when the end token is generated
        if sampled_token_id == end_token_id:
            break

        # Convert token ID to a word
        sampled_word = french_tokenizer.index_word.get(
            sampled_token_id,
            "<unk>"
        )

        # Skip padding and special start tokens
        if sampled_token_id != 0 and sampled_token_id != start_token_id:
            translated_words.append(sampled_word)

        # Use the predicted token for the next step
        target_seq = np.array([[sampled_token_id]])

        # Update the decoder states
        states_value = [h, c]

    return " ".join(translated_words)


# ------------------------------------------------------------
# 12. TEST THE MODEL
# ------------------------------------------------------------

test_sentences = [
    "hello",
    "thank you",
    "good night",
    "i am fine",
    "goodbye",
    "open the door",
    "close the door"
]

print("\n========== TRANSLATION RESULTS ==========\n")

for sentence in test_sentences:

    translation = translate_sentence(sentence)

    print("English:", sentence)
    print("French:", translation)
    print("-" * 40)


# ------------------------------------------------------------
# 13. TRANSLATE YOUR OWN SENTENCE
# ------------------------------------------------------------

sentence = input("Enter an English sentence: ")

translation = translate_sentence(sentence)

print("English:", sentence)
print("French translation:", translation)

Number of training examples: 20
English vocabulary size: 36
French vocabulary size: 39
Encoder input shape: (20, 4)
Decoder input shape: (20, 5)
Decoder target shape: (20, 5)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 64)  │      2,304 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 64)  │      2,496 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     98,816 │ decoder_embeddin… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_prediction     │ (None, None, 39)  │      5,031 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 207,463 (810.40 KB)

 Trainable params: 207,463 (810.40 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - loss: 3.6535 - sparse_categorical_accuracy: 0.1685   
Epoch 2/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 3.6140 - sparse_categorical_accuracy: 0.4045
Epoch 3/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.5491 - sparse_categorical_accuracy: 0.3933
Epoch 4/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.4047 - sparse_categorical_accuracy: 0.3820
Epoch 5/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.0763 - sparse_categorical_accuracy: 0.2697
Epoch 6/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.6704 - sparse_categorical_accuracy: 0.2360
Epoch 7/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.6402 - sparse_categorical_accuracy: 0.3820
Epoch 8/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.5254 - sparse_categorical_accuracy: 0.3820
Epoch 9/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.4295 - sparse_categorical_accuracy: 0.4270
Epoch 10/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.3780 - sp